# Week 13: End-to-End Machine Learning Pipeline

In this notebook, we build a complete ML pipeline predicting Wine Quality and compare three models: Logistic Regression, Random Forest, and XGBoost.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import os

### 1. Load Data

In [ ]:
# Load the wine quality dataset
path = '../projects/10_Wine_Quality_Classifier/data/winequality-red.csv'
df = pd.read_csv(path, sep=';')

# Binarize target
df['quality_label'] = (df['quality'] > 5).astype(int)
X = df.drop(['quality', 'quality_label'], axis=1)
y = df['quality_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data loaded and scaled!")

### 2. Train Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
    "XGBoost": XGBClassifier(learning_rate=0.1, max_depth=3, use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "CM": confusion_matrix(y_test, y_pred)
    }
    print(f"{name} trained.")

### 3. Evaluate and Visualize

In [ ]:
metrics_df = pd.DataFrame({
    "Model": list(results.keys()),
    "Accuracy": [r["Accuracy"] for r in results.values()],
    "Precision": [r["Precision"] for r in results.values()],
    "Recall": [r["Recall"] for r in results.values()],
    "F1-Score": [r["F1-Score"] for r in results.values()]
})

metrics_melted = pd.melt(metrics_df, id_vars=["Model"], var_name="Metric", value_name="Score")

plt.figure(figsize=(10, 4))
sns.barplot(data=metrics_melted, x="Metric", y="Score", hue="Model", palette="mako")
plt.ylim(0.6, 1.0)
plt.title("Head-to-Head Model Performance")
plt.show()